In [0]:
# Configuration parameters - can be overridden via widgets or job parameters
dbutils.widgets.text("environment", "dev", "Environment (dev/staging/prod)")
dbutils.widgets.text("storage_account", "<storage-account>", "Storage Account Name")
dbutils.widgets.text("container", "unity-catalog-storage", "Container Name")
dbutils.widgets.dropdown("use_autoloader", "true", ["true", "false"], "Use Auto Loader")

# Get parameter values
env = dbutils.widgets.get("environment")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
use_autoloader = dbutils.widgets.get("use_autoloader") == "true"

# Construct paths
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{env}/silver"
checkpoint_base = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{env}/checkpoints"

print(f"Environment: {env}")
print(f"Base path: {base_path}")
print(f"Checkpoint base: {checkpoint_base}")
print(f"Auto Loader enabled: {use_autoloader}")

In [0]:
files = dbutils.fs.ls(base_path)

print(f"Found {len(files)} files/directories in silver layer:")
for file in files:
    print(f"  - {file.name}")

In [0]:
from pyspark.sql.functions import input_file_name, current_timestamp
import re

# Track loaded tables
loaded_tables = {}
errors = []

if use_autoloader:
    print("Using Auto Loader for incremental processing")
    print("="*80)
    
    for file in files:
        table_path = file.path
        table_name = file.name.rstrip('/').replace('.', '_').replace('-', '_')
        checkpoint_path = f"{checkpoint_base}/bronze_load/{table_name}"
        
        try:
            # Check if directory is a Delta table (has _delta_log)
            delta_log_path = f"{table_path.rstrip('/')}/_delta_log"
            is_delta = False
            try:
                delta_log_files = dbutils.fs.ls(delta_log_path)
                is_delta = len(delta_log_files) > 0
            except:
                is_delta = False

            if is_delta:
                # Read Delta table directly (do NOT use Auto Loader)
                df = spark.read.format("delta").load(table_path)
                row_count = df.count()
                col_count = len(df.columns)
                globals()[table_name] = df
                loaded_tables[table_name] = {
                    'path': table_path,
                    'rows': row_count,
                    'columns': col_count,
                    'method': 'delta'
                }
                print(f"✓ Loaded {table_name}: {row_count:,} rows, {col_count} columns (Delta)")
                display(df.limit(5))
            else:
                # Use Auto Loader for non-Delta directories
                stream_df = (spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "delta")
                    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
                    .option("cloudFiles.inferColumnTypes", "true")
                    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                    .load(table_path))
                
                temp_path = f"{checkpoint_base}/temp/{table_name}"
                query = (stream_df
                    .writeStream
                    .format("delta")
                    .option("checkpointLocation", checkpoint_path)
                    .option("mergeSchema", "true")
                    .trigger(availableNow=True)
                    .start(temp_path))
                query.awaitTermination()
                df = spark.read.format("delta").load(temp_path)
                row_count = df.count()
                col_count = len(df.columns)
                globals()[table_name] = df
                loaded_tables[table_name] = {
                    'path': table_path,
                    'rows': row_count,
                    'columns': col_count,
                    'method': 'autoloader'
                }
                print(f"✓ Loaded {table_name}: {row_count:,} rows, {col_count} columns (Auto Loader)")
                display(df.limit(5))
            
        except Exception as e:
            error_msg = f"✗ Failed to load {table_name}: {str(e)}"
            print(error_msg)
            errors.append({'table': table_name, 'error': str(e)})
            
else:
    print("Using standard batch read (no Auto Loader)")
    print("="*80)
    
    for file in files:
        table_path = file.path
        table_name = file.name.rstrip('/').replace('.', '_').replace('-', '_')
        
        try:
            df = spark.read.format("delta").load(table_path)
            row_count = df.count()
            col_count = len(df.columns)
            globals()[table_name] = df
            loaded_tables[table_name] = {
                'path': table_path,
                'rows': row_count,
                'columns': col_count,
                'method': 'batch'
            }
            print(f"✓ Loaded {table_name}: {row_count:,} rows, {col_count} columns (Batch)")
            display(df.limit(5))
        except Exception as e:
            error_msg = f"✗ Failed to load {table_name}: {str(e)}"
            print(error_msg)
            errors.append({'table': table_name, 'error': str(e)})

print(f"\n{'='*80}")
print(f"Summary: Loaded {len(loaded_tables)}/{len(files)} tables successfully")
if errors:
    print(f"Errors: {len(errors)} tables failed to load")
    for err in errors:
        print(f"  - {err['table']}: {err['error']}")


# Gold Layer - Business Aggregations

**Purpose**: Create business-ready aggregated tables from Silver layer data.

**Approach**: 
* Read from Silver Delta tables (no Auto Loader needed)
* Apply business logic and aggregations
* Write to Gold Delta tables
* Use batch processing (data is small and updates are infrequent)

**Why Not Auto Loader?**
* Auto Loader is for ingesting **files** from cloud storage (Bronze layer)
* Silver tables are already **Delta format** - use Delta reads instead
* Your data is small and batch-loaded - streaming adds unnecessary complexity

In [0]:
# Simple batch read from Silver Delta tables
print("Loading Silver layer tables...")
print("="*80)

# Define Silver table paths
silver_tables = {
    'forecast': f"{base_path}/forecast_silver",
    'historical': f"{base_path}/historical_silver",
    'global_weather': f"{base_path}/global_weather_silver",
    'normal': f"{base_path}/normal_silver"
}

# Load each table
for table_name, table_path in silver_tables.items():
    try:
        df = spark.read.format("delta").load(table_path)
        row_count = df.count()
        col_count = len(df.columns)
        
        # Store in globals with _silver suffix
        globals()[f"{table_name}_silver"] = df
        
        print(f"✓ {table_name}_silver: {row_count:,} rows, {col_count} columns")
        
    except Exception as e:
        print(f"✗ Failed to load {table_name}: {str(e)}")

print("\n" + "="*80)
print("✓ Silver tables loaded successfully")
print("="*80)

In [0]:
from pyspark.sql.functions import col, count, avg, max, min, countDistinct

print("Creating Gold table: country_weather_summary")
print("="*80)

# Aggregate forecast data by country
country_summary = forecast_silver.groupBy("country_code") \
    .agg(
        countDistinct("city_name").alias("city_count"),
        count("*").alias("forecast_count"),
        avg("precipitation_probability").alias("avg_precip_prob"),
        avg("rain_probability").alias("avg_rain_prob"),
        avg("snow_probability").alias("avg_snow_prob"),
        max("precipitation_probability").alias("max_precip_prob")
    ) \
    .orderBy(col("forecast_count").desc())

print(f"Rows: {country_summary.count():,}")
display(country_summary)

In [0]:
from pyspark.sql.functions import col, count, avg, max, min, stddev

print("Creating Gold table: station_temperature_stats")
print("="*80)

# Aggregate weather station data
station_stats = global_weather_silver.groupBy("station") \
    .agg(
        count("*").alias("measurement_count"),
        avg("tmax").alias("avg_tmax"),
        avg("tmin").alias("avg_tmin"),
        max("tmax").alias("max_tmax"),
        min("tmin").alias("min_tmin"),
        avg("prcp").alias("avg_precipitation"),
        max("prcp").alias("max_precipitation")
    ) \
    .orderBy(col("measurement_count").desc())

print(f"Rows: {station_stats.count():,}")
display(station_stats)

In [0]:
# Define Gold layer output path
gold_base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{env}/gold"

print("\n" + "="*80)
print("WRITING GOLD TABLES")
print("="*80)
print(f"Gold layer path: {gold_base_path}\n")

def write_gold_table(df, table_name, description):
    """
    Write Gold table with standard options
    """
    output_path = f"{gold_base_path}/{table_name}"
    
    print(f"Writing: {table_name}")
    print(f"  Description: {description}")
    print(f"  Rows: {df.count():,}")
    print(f"  Path: {output_path}")
    
    try:
        (df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("delta.autoOptimize.optimizeWrite", "true")
            .option("delta.autoOptimize.autoCompact", "true")
            .save(output_path))
        
        print(f"  ✓ Success\n")
        return True
    except Exception as e:
        print(f"  ✗ Failed: {str(e)}\n")
        return False

# Write the gold tables
results = []
results.append(write_gold_table(
    country_summary, 
    "country_weather_summary",
    "Weather forecast statistics aggregated by country"
))

results.append(write_gold_table(
    station_stats,
    "station_temperature_stats",
    "Temperature and precipitation statistics by weather station"
))

print("="*80)
print(f"Summary: {sum(results)}/{len(results)} tables written successfully")
print("="*80)